In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="hf_pipeline_mrpc_short_maxlen_64",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [2]:
model_name = "textattack/distilbert-base-uncased-MRPC"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
)

id2label = {int(k): v for k, v in model.config.id2label.items()}
label2id = {v: k for k, v in id2label.items()}

print(model_name)
print(id2label)
print("pipeline_device:", clf.model.device)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

textattack/distilbert-base-uncased-MRPC
{0: 'LABEL_0', 1: 'LABEL_1'}
pipeline_device: mps:0


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"], dtype=int)

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
pair_inputs = [{"text": s1, "text_pair": s2} for s1, s2 in zip(sent1, sent2)]
max_length = 64
batch_size = 64

print(pair_inputs[0])
print("max_length:", max_length)
print("batch_size:", batch_size)


{'text': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'text_pair': '" The foodservice pie business does not fit our long-term growth strategy .'}
max_length: 64
batch_size: 64


In [5]:
outputs = []

with torch.no_grad():
    for i in tqdm(range(0, len(pair_inputs), batch_size)):
        batch_inputs = pair_inputs[i:i + batch_size]
        batch_outputs = clf(
            batch_inputs,
            batch_size=batch_size,
            truncation=True,
            max_length=max_length,
            padding=True,
            function_to_apply="softmax",
        )
        outputs.extend(batch_outputs)

pred_labels = [out["label"] for out in outputs]
pred_scores = np.array([float(out["score"]) for out in outputs], dtype=float)
y_pred = np.array([label2id[label] for label in pred_labels], dtype=int)

print("done")
print("num_outputs:", len(outputs))


  0%|          | 0/7 [00:00<?, ?it/s]

done
num_outputs: 408


In [ ]:

vault.create_record_list("pipeline_distilbert_shortened", column_names=["prediction", "score"])

for i in range(len(y_pred)):
    vault.append_record("pipeline_distilbert_shortened", 
                        {
                            "prediction": y_pred[i],
                            "score": pred_scores[i] ,
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT pipeline_distilbert_shortened"
embedding = get_embeddings(description)
vault.create_description("pipeline_distilbert_shortened", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("pipeline_distilbert_shortened", cat, embedding, prop)

In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
target_names = [id2label[i] for i in sorted(id2label)]
report = classification_report(y_true, y_pred, target_names=target_names)
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=target_names))


{'accuracy': 0.8455882352941176, 'f1': 0.8944723618090452}
              precision    recall  f1-score   support

     LABEL_0       0.87      0.60      0.71       129
     LABEL_1       0.84      0.96      0.89       279

    accuracy                           0.85       408
   macro avg       0.85      0.78      0.80       408
weighted avg       0.85      0.85      0.84       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", pred_labels[i], "score:", float(pred_scores[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 label: LABEL_1 score: 0.983514666557312
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 label: LABEL_0 score: 0.8176295757293701
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 1 label: LABEL_1 score: 0.9455713629722595
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The 

In [8]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", pred_labels[i], "score:", float(pred_scores[i]))


num_errors: 63
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 1 label: LABEL_1 score: 0.9455713629722595
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
true: 0 pred: 1 label: LABEL_1 score: 0.929533839225769
idx: 26
sentence1: Cooley said he expects Muhammad will similarly be called as a witness at a pretrial hearing for Malvo .
sentence2: Lee Boyd Malvo will be called as a witness Wednesday in a pretrial hearing for fellow sniper suspect John Allen Muhammad .
true: 0 pred: 1 label: LABEL_1 score: 0.8425254225730896
idx: 35
senten

In [9]:

vault.create_record_list("hf_pipeline_mrpc_short_maxlen_64_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("hf_pipeline_mrpc_short_maxlen_64_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "pipeline_distilbert_shortened": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT hf_pipeline_mrpc_short_maxlen_64"
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_mrpc_short_maxlen_64", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_mrpc_short_maxlen_64", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'textattack/distilbert-base-uncased-MRPC',
 'device': 'mps',
 'max_length': 64,
 'num_examples': 408,
 'accuracy': 0.8455882352941176,
 'f1': 0.8944723618090452}

In [ ]:
description = "INSERT TEXT HERE ABOUT hf_pipeline_mrpc_short_maxlen_64" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_mrpc_short_maxlen_64", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_mrpc_short_maxlen_64", cat, embedding, prop)